# Reproduce SPMB 2026 — end-to-end from the shipped CSVs

This notebook walks through the **Tier A** reproduction (no TUH access
needed). At the end you should have:

* Every paper figure regenerated under `../figures/` (except the two
  Magna case-study panels — see `figures/captions.md`).
* Every paper statistic recomputed under
  `../data/advanced_stats_summary.md`.
* Confirmation that the key headline numbers (Table 1 + Table 2 in the
  paper) match this repo's CSVs to ≤ 0.005 absolute AUROC.

Total wall time: ~3 minutes on a laptop.

## 0. Imports and paths

In [ ]:
import sys, subprocess
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = REPO / 'src'
DATA = REPO / 'data'
FIGS = REPO / 'figures'
sys.path.insert(0, str(SRC))

print('repo:', REPO)
print('python:', sys.version.split()[0])

## 1. Quick sanity check on the shipped data

In [ ]:
import pandas as pd, json

sweep = pd.read_csv(DATA / 'sweep_latest.csv')
preds = pd.read_csv(DATA / 'per_recording_predictions_latest.csv')
manifest = json.loads((DATA / 'manifest_tuab_eval.json').read_text())

print('sweep_latest.csv rows :', len(sweep))
print('per-rec rows         :', len(preds))
print('manifest n_recordings:', manifest['n_recordings'])
print('manifest n_patients  :', manifest['n_patients'])
print('normal/abnormal      :',
      manifest['n_normal'], '/', manifest['n_abnormal'])
sweep.head()

## 2. Reproduce Table 1 — the headline paper numbers

Read directly from `sweep_latest.csv`.

In [ ]:
def headline(axis, severity, arm):
    row = sweep[(sweep['axis']==axis) &
                (sweep['severity'].astype(str)==str(severity)) &
                (sweep['arm']==arm)].iloc[0]
    return f'{row.auroc:.3f} [{row.ci_lo:.3f}, {row.ci_hi:.3f}]'

print('Clean baseline AUROC   :', headline('sampling_rate', 250, 'naive'))
print('Power-line 30 uV naive :', headline('power_line', 30, 'naive'))
print('Power-line 30 uV canon :', headline('power_line', 30, 'canonicalized'))
print('Power-line 60 uV naive :', headline('power_line', 60, 'naive')
      if not sweep[(sweep['axis']=='power_line') &
                   (sweep['severity'].astype(str)=='60.0')].empty
      else 'n/a — not in shipped CSV')

## 3. Run the full advanced-statistics battery

This computes FDR, Bayesian beta-binomial, mixed-effects logistic,
AURC, conformal coverage, mutual information, and the paper-number
verification block. ~1 minute.

In [ ]:
r = subprocess.run(
    [sys.executable, str(SRC / 'run_all_advanced_stats.py')],
    capture_output=True, text=True
)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
    raise RuntimeError('run_all_advanced_stats.py failed')

In [ ]:
summary = (DATA / 'advanced_stats_summary.md').read_text()
print(summary[:4000])

## 4. Regenerate every figure (~30 s)

In [ ]:
r = subprocess.run(
    [sys.executable, str(SRC / 'paper_figures.py')],
    capture_output=True, text=True
)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
    raise RuntimeError('paper_figures.py failed')

In [ ]:
from IPython.display import IFrame
IFrame(str(FIGS / 'fig_axes.pdf'), width=720, height=540)

## 5. Run the unit tests

In [ ]:
r = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q',
     str(SRC / 'test_perturbations.py'),
     str(SRC / 'test_statistical_tests.py'),
     str(SRC / 'test_tuev_labels.py')],
    capture_output=True, text=True, cwd=str(REPO)
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)

## 6. Compare your run to the paper's headline numbers

The verification block inside `advanced_stats_results.json` re-derives
every paper-cited AUROC from the per-recording CSV and flags any drift
above 0.005 absolute.

In [ ]:
results = json.loads((DATA / 'advanced_stats_results.json').read_text())
ver = results['verification']
print(f"drift_count: {ver['drift_count']}")
for chk in ver['checks']:
    flag = '  ' if not chk['drift'] else 'XX'
    print(f"{flag} {chk['name']:40s} paper={chk['paper_value']!s:>10}  "
          f"computed={chk['computed_value']!s:>10}")

If `drift_count == 0`, every paper number matches this repo's CSVs.
If non-zero, the offending rows print with `XX` — please file an
issue with your numpy / scipy / onnxruntime versions.